# Chapter 13: Diagnosing and Steering

## Applying Geometric Algebra to context-ignoring detection, hallucination diagnostics, and plane-specific steering

We have now built a complete toolkit for understanding transformers through Geometric Algebra: rotors, bivectors, holonomy curvature, commutator structure, dependency profiles, and capacity metrics. In this final applications chapter, we put all of these tools to work on three practical diagnostic tasks:

1. **Context-ignoring detection** — Does the model actually use the context it was given, or does it fall back on memorized knowledge? We compare GA profiles with and without context, and show that curvature *plane* shifts are more informative than magnitude changes alone.

2. **Hallucination signatures** — When a model hallucinates, it generates confident-sounding text that is not grounded in anything. We look for layers that are geometrically "busy but useless": high curvature magnitude but low dependency on the final output. The GA question is: are these busy layers rotating in *consistent* planes (systematic bias) or *random* planes (noise)?

3. **Steering targets** — If we want to intervene on a model's behavior at a specific layer, GA tells us *which plane* to intervene in. Instead of modifying all directions equally (a rank-$k$ intervention), we target only the principal rotation plane (a rank-2 intervention in $k$-dimensional space).

Each application uses the full pipeline: `ltg_ga.analyse()` for extraction, bivector analysis for planes, holonomy for curvature, and dependency for functional relevance.

In [ ]:
import sys, os

# Ensure the project root is on the path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import ltg_ga
from layer_time_ga.curvature import holonomy_rotor, commutator_field

print("Imports OK")
print(f"NumPy {np.__version__}")

In [ ]:
# Load model
model = ltg_ga.load_model("Qwen/Qwen2.5-7B")
print(f"Model: {model.name}")
print(f"Layers: {model.n_layers}, Hidden dim: {model.hidden_dim}")
print(f"Device: {model.device}")

## Application 1: Does the Model Use Context?

When we give a language model a factual question, it can answer from two sources: **parametric knowledge** (memorized during training) or **contextual knowledge** (information provided in the prompt). A critical diagnostic question is: *did the model actually attend to the context, or did it ignore the context and rely on memorization?*

**The scalar approach** would compare curvature magnitudes between with-context and without-context prompts. If the curvature profile changes, something is different. But *how much* does it change? A 5% magnitude change could be noise.

**The GA approach** adds a crucial dimension: we compare not just *how much* the model curves, but *in which planes* it curves. The key insight is:

> **A curvature plane shift is a stronger indicator of context use than a curvature magnitude change.**

If the model ignores the context, its rotation planes should be nearly identical with and without context — it is doing the same internal computation either way. But if the model *uses* the context, the rotation planes should shift: the model is routing information through different subspaces to integrate the new knowledge.

Let us test this with a factual question, presented with and without relevant context.

In [ ]:
# --- Two prompts: with and without context ---

# WITHOUT context: the model must rely on parametric memory
prompt_no_ctx = "The largest ocean on Earth is"

# WITH context: we provide a relevant passage
prompt_with_ctx = (
    "The Pacific Ocean covers approximately 165.25 million square kilometers, "
    "making it larger than all of Earth's land area combined. "
    "The largest ocean on Earth is"
)

# Analyse both (disable dependency for speed in this comparison)
result_no_ctx = ltg_ga.analyse(prompt_no_ctx, model=model, compute_dependency=False)
result_with_ctx = ltg_ga.analyse(prompt_with_ctx, model=model, compute_dependency=False)

print("=== Without Context ===")
print(f"  Prompt: \"{prompt_no_ctx}\"")
print(f"  Tokens: {result_no_ctx.n_tokens}, Layers: {result_no_ctx.n_layers}")
print(f"  Mean curvature: {result_no_ctx.curvature_by_layer.mean():.4f}")
print(f"  Peak curvature layer: {result_no_ctx.curvature_by_layer.argmax()}")
print()
print("=== With Context ===")
print(f"  Prompt: \"{prompt_with_ctx[:60]}...\"")
print(f"  Tokens: {result_with_ctx.n_tokens}, Layers: {result_with_ctx.n_layers}")
print(f"  Mean curvature: {result_with_ctx.curvature_by_layer.mean():.4f}")
print(f"  Peak curvature layer: {result_with_ctx.curvature_by_layer.argmax()}")

# --- Side-by-side curvature profiles ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: curvature-by-layer comparison
ax1.plot(result_no_ctx.curvature_by_layer, color='#4477AA', linewidth=2,
         label='No context', marker='o', markersize=3)
ax1.plot(result_with_ctx.curvature_by_layer, color='#EE6677', linewidth=2,
         label='With context', marker='s', markersize=3)
ax1.set_xlabel('Layer')
ax1.set_ylabel('Mean holonomy curvature')
ax1.set_title('Curvature Magnitude: With vs Without Context')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: magnitude difference
min_len = min(len(result_no_ctx.curvature_by_layer),
              len(result_with_ctx.curvature_by_layer))
diff = (result_with_ctx.curvature_by_layer[:min_len]
        - result_no_ctx.curvature_by_layer[:min_len])
ax2.bar(range(min_len), diff, color=['#228833' if d > 0 else '#AA3377' for d in diff],
        edgecolor='white', linewidth=0.5)
ax2.axhline(0, color='grey', linewidth=0.5)
ax2.set_xlabel('Layer')
ax2.set_ylabel('$\\Delta$ curvature (with - without)')
ax2.set_title('Curvature Difference by Layer')
ax2.grid(True, axis='y', alpha=0.3)

fig.tight_layout()
plt.show()
print("Magnitude differences are informative but noisy. Let us examine the PLANES.")

In [ ]:
# --- Compare holonomy bivectors at the peak curvature layer ---
# The bivector tells us WHICH PLANE the curvature lives in.
# If the planes differ, the model is doing different computations.

# Find the peak curvature layer for each result
peak_no_ctx = int(result_no_ctx.curvature_by_layer.argmax())
peak_with_ctx = int(result_with_ctx.curvature_by_layer.argmax())
print(f"Peak curvature layer (no context):   {peak_no_ctx}")
print(f"Peak curvature layer (with context): {peak_with_ctx}")
print()

# Compare bivectors at a shared layer (use the no-context peak for fair comparison)
compare_layer = peak_no_ctx

# Extract the bivector (rotation generator) at the comparison layer
biv_no_ctx = result_no_ctx.bivector_field[min(compare_layer, len(result_no_ctx.bivector_field) - 1)]
biv_with_ctx = result_with_ctx.bivector_field[min(compare_layer, len(result_with_ctx.bivector_field) - 1)]

# Cosine similarity between the two bivector matrices (flattened)
# This measures alignment of the ROTATION PLANES, not magnitudes
vec_a = biv_no_ctx.matrix.flatten()
vec_b = biv_with_ctx.matrix.flatten()
cos_sim = np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b) + 1e-12)

print(f"=== Bivector Comparison at Layer {compare_layer} ===")
print(f"  Bivector norm (no context):   {biv_no_ctx.norm:.4f}")
print(f"  Bivector norm (with context): {biv_with_ctx.norm:.4f}")
print(f"  Norm ratio:                   {biv_with_ctx.norm / (biv_no_ctx.norm + 1e-12):.3f}")
print(f"  Cosine similarity (planes):   {cos_sim:.4f}")
print()

# Interpret the result
if abs(cos_sim) > 0.9:
    print("INTERPRETATION: High cosine similarity -> the model is rotating in")
    print("  nearly the SAME plane with and without context. This suggests the")
    print("  model may be IGNORING the context at this layer.")
elif abs(cos_sim) > 0.5:
    print("INTERPRETATION: Moderate cosine similarity -> the model is partially")
    print("  shifting its rotation plane when context is added. The context is")
    print("  influencing computation but not dominating it.")
else:
    print("INTERPRETATION: Low cosine similarity -> the model is rotating in a")
    print("  DIFFERENT plane when context is provided. This is strong evidence")
    print("  that the model is USING the context to route information through")
    print("  different subspaces.")

print()
print("NOTE: This plane-based diagnostic is more informative than magnitude alone.")
print("Two prompts can have similar curvature magnitudes but very different planes,")
print("revealing fundamentally different internal computations.")

### The Cayley Bivector: Measuring Context Influence Directly

The bivector comparison above uses the *rotor field* bivectors (which come from the layer transition operators). A more direct measurement uses the **Cayley bivector**, which encodes the rotation mapping the *prior* representation (query alone) to the *posterior* representation (query + context) at each layer.

For normalised vectors $u_\pi$ (prior) and $v_q$ (posterior):

$$A(u_\pi, v_q) = \frac{v_q u_\pi^\top - u_\pi v_q^\top}{1 + v_q^\top u_\pi}$$

The **steering magnitude** $\tau = \|A\|_F = \tan(\theta/2)$ quantifies how much the context rotates the representation. If $\tau < 0.05$, the model is "vibing" — ignoring context.

In [ ]:
# --- Cayley Bivector: direct measurement of context influence ---
import layer_time_geometry as core
from layer_time_ga.algebra import bivector_from_skew

prompt_bare = "What is the capital of France"
prompt_ctx = ("Paris is the largest city in France and serves as its capital. "
              "What is the capital of France")

H_bare = core.extract_hidden_states(model.hf_model, model.tokenizer,
                                     prompt_bare, model.device)
H_ctx = core.extract_hidden_states(model.hf_model, model.tokenizer,
                                    prompt_ctx, model.device)

print(f"Bare prompt: {H_bare.shape[1]} tokens, Context prompt: {H_ctx.shape[1]} tokens")
print()

# Compute Cayley bivector at each layer (last-token hidden state)
n_layers = H_bare.shape[0]
taus = np.zeros(n_layers)
thetas = np.zeros(n_layers)
cosines = np.zeros(n_layers)

for l in range(n_layers):
    u = H_bare[l, -1].cpu().numpy().astype(np.float64)
    v = H_ctx[l, -1].cpu().numpy().astype(np.float64)
    u = u / (np.linalg.norm(u) + 1e-12)
    v = v / (np.linalg.norm(v) + 1e-12)
    
    # Cayley bivector
    denom = 1.0 + np.dot(v, u)
    A = (np.outer(v, u) - np.outer(u, v)) / (denom + 1e-12)
    taus[l] = np.linalg.norm(A, 'fro')
    thetas[l] = np.degrees(np.arccos(np.clip(np.dot(u, v), -1, 1)))
    cosines[l] = np.dot(u, v)

# Print table
print(f"{'Layer':>5s} {'tau':>8s} {'theta (deg)':>12s} {'cos(u,v)':>10s}")
print("-" * 40)
for l in [0, 7, 14, int(taus.argmax()), 21, 27]:
    if l < n_layers:
        marker = " <-- peak" if l == taus.argmax() else ""
        print(f"{l:5d} {taus[l]:8.4f} {thetas[l]:12.1f} {cosines[l]:10.4f}{marker}")

print(f"\nPeak tau at layer {taus.argmax()} ({taus.max():.4f})")
print(f"Mean tau: {taus.mean():.4f}")
print(f"All taus > 0.05 (vibe threshold): {(taus[1:] > 0.05).all()}")
print("=> The model IS using context (not vibing)")

# Plot tau profile
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(n_layers), taus, color='#2E6DAD', linewidth=2, marker='o', markersize=3)
ax.fill_between(range(n_layers), taus, alpha=0.15, color='#2E6DAD')
ax.axhline(y=0.05, color='red', linestyle='--', alpha=0.5, label=r'$\tau = 0.05$ (vibe threshold)')
ax.axvline(taus.argmax(), color='grey', linestyle='--', alpha=0.5,
           label=f'Peak = layer {taus.argmax()}')
ax.set_xlabel('Layer', fontsize=11)
ax.set_ylabel(r'Steering magnitude $\tau$', fontsize=11)
ax.set_title('Cayley Bivector Magnitude: Bare vs Context Prompt', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Principal planes at peak layer
peak_l = int(taus.argmax())
u = H_bare[peak_l, -1].cpu().numpy().astype(np.float64)
v = H_ctx[peak_l, -1].cpu().numpy().astype(np.float64)
u = u / (np.linalg.norm(u) + 1e-12)
v = v / (np.linalg.norm(v) + 1e-12)
A_peak = (np.outer(v, u) - np.outer(u, v)) / (1.0 + np.dot(v, u) + 1e-12)
B_peak = bivector_from_skew(A_peak)
planes = B_peak.principal_planes(n_planes=3)
print(f"\nCayley bivector principal planes at peak layer {peak_l}:")
for i, p in enumerate(planes):
    if p['weight'] > 1e-10:
        print(f"  Plane {i}: angle = {p['angle']:.4f} rad, weight = {p['weight']:.4f}")

## Application 2: Hallucination Signatures

When a language model *hallucinates*, it generates plausible-sounding text that is factually incorrect. From a geometric perspective, what does this look like internally?

The hypothesis is that hallucination involves **busy-but-useless layers**: layers with high curvature (lots of geometric activity — the representation is being rotated aggressively) but low dependency (this rotation does not meaningfully contribute to the final output). The model is *doing work* at these layers, but the work is not *functional*.

Scalar diagnostics can identify such layers by comparing curvature magnitude to dependency magnitude. But GA adds a critical follow-up question:

> **Are the busy-but-useless layers rotating in consistent planes or random planes?**

- **Consistent planes** (high pairwise bivector similarity): The model is applying a systematic but irrelevant transformation — a learned bias that activates regardless of input. This is a *structured* failure mode.
- **Random planes** (low pairwise bivector similarity): The model's rotation at these layers is essentially noise — no consistent pattern. This is an *unstructured* failure mode.

The distinction matters for intervention: structured hallucination can potentially be corrected by suppressing a specific plane, while unstructured hallucination may require more fundamental changes.

Let us run a full analysis (with dependency) and identify the busy-but-useless layers.

In [ ]:
# --- Full analysis with dependency to detect busy-but-useless layers ---
prompt_halluc = "The inventor of the telephone was Alexander Graham Bell, who"
result = ltg_ga.analyse(prompt_halluc, model=model, compute_dependency=True)
result.summary()

# --- Identify busy-but-useless layers ---
# "Busy" = high curvature (top quartile)
# "Useless" = low dependency (bottom quartile)

curv = result.curvature_by_layer
dep = result.dependency_profile

if dep is not None:
    # Align dimensions: curvature has L-1 entries, dependency has L entries
    # Use the first len(curv) entries of dependency for alignment
    dep_aligned = dep[:len(curv)]
    
    # Normalize both to [0, 1] for fair comparison
    curv_norm = (curv - curv.min()) / (curv.max() - curv.min() + 1e-12)
    dep_norm = (dep_aligned - dep_aligned.min()) / (dep_aligned.max() - dep_aligned.min() + 1e-12)
    
    # Busy-but-useless: high curvature AND low dependency
    curv_threshold = np.percentile(curv_norm, 75)  # top 25% curvature
    dep_threshold = np.percentile(dep_norm, 25)     # bottom 25% dependency
    
    busy_useless = np.where((curv_norm >= curv_threshold) & (dep_norm <= dep_threshold))[0]
    
    print(f"\n=== Busy-but-Useless Layer Detection ===")
    print(f"Curvature threshold (75th pct): {curv_threshold:.4f}")
    print(f"Dependency threshold (25th pct): {dep_threshold:.4f}")
    print(f"Candidate layers: {busy_useless.tolist()}")
    print()
    
    # For these layers, extract principal rotation planes and check consistency
    if len(busy_useless) >= 2:
        # Extract bivectors for busy-but-useless layers
        bu_bivectors = [result.bivector_field[min(l, len(result.bivector_field) - 1)]
                        for l in busy_useless]
        
        # Compute pairwise bivector similarity (cosine between flattened matrices)
        n_bu = len(bu_bivectors)
        similarity_matrix = np.zeros((n_bu, n_bu))
        for i in range(n_bu):
            for j in range(n_bu):
                vi = bu_bivectors[i].matrix.flatten()
                vj = bu_bivectors[j].matrix.flatten()
                similarity_matrix[i, j] = (np.dot(vi, vj) /
                    (np.linalg.norm(vi) * np.linalg.norm(vj) + 1e-12))
        
        # Average off-diagonal similarity
        mask = ~np.eye(n_bu, dtype=bool)
        mean_sim = np.abs(similarity_matrix[mask]).mean()
        
        print(f"=== Plane Consistency Analysis ===")
        print(f"Number of busy-but-useless layers: {n_bu}")
        print(f"Mean pairwise |cosine similarity|: {mean_sim:.4f}")
        print()
        
        if mean_sim > 0.5:
            print("DIAGNOSIS: HIGH consistency -> Systematic rotation pattern.")
            print("  These layers apply a coherent but non-functional transformation.")
            print("  This is a STRUCTURED hallucination signature.")
            print("  Intervention: suppress the dominant plane at these layers.")
        elif mean_sim > 0.2:
            print("DIAGNOSIS: MODERATE consistency -> Partially structured pattern.")
            print("  Some shared rotation structure, but not fully coherent.")
        else:
            print("DIAGNOSIS: LOW consistency -> Random rotation pattern.")
            print("  These layers have no coherent rotation structure.")
            print("  This is an UNSTRUCTURED noise signature.")
            print("  Intervention: pruning or regularization may be more appropriate.")
        
        # Visualize the similarity matrix
        fig, ax = plt.subplots(figsize=(5, 4))
        im = ax.imshow(np.abs(similarity_matrix), cmap='YlOrRd', vmin=0, vmax=1)
        ax.set_xticks(range(n_bu))
        ax.set_yticks(range(n_bu))
        ax.set_xticklabels([f'L{l}' for l in busy_useless])
        ax.set_yticklabels([f'L{l}' for l in busy_useless])
        ax.set_title('|Bivector Cosine Similarity|\namong Busy-but-Useless Layers')
        plt.colorbar(im, ax=ax)
        fig.tight_layout()
        plt.show()
    else:
        print(f"Only {len(busy_useless)} busy-but-useless layer(s) found.")
        print("Need at least 2 for pairwise plane comparison.")
        print("This prompt may not exhibit a strong hallucination signature.")
else:
    print("Dependency not available. Re-run with compute_dependency=True.")

### The Binet–Cauchy Cosine: Detecting Causal Inversions

Bivector coherence tells you whether reasoning *paths* agree. But a subtler failure
is when the logical *direction* of reasoning reverses — a conclusion appears before
its premise. This is a **causal inversion**.

The **Binet–Cauchy cosine** detects this by comparing the *oriented areas* of two planes:
- The model's reasoning trajectory plane: $u_i \wedge u_{i+1}$
- The expected information flow plane: $c_q \wedge c_{\text{ctx}}$

$$\text{BCcos} = \frac{\det S}{\|u_i \wedge u_{i+1}\| \cdot \|c_q \wedge c_{\text{ctx}}\| + \epsilon}$$

where $S$ is the 2×2 matrix of inner products. The determinant $\det S$ is the
contraction of two bivectors — a signed oriented area that is **positive** when the
planes are aligned and **negative** when they are reversed (causal inversion).

In [ ]:
# ── Binet-Cauchy cosine: detect causal inversions ────────────────
from layer_time_ga.algebra import binet_cauchy_cosine

# Use the hidden states from factual vs confabulation prompts
# Compare consecutive layer embeddings against a reference direction
H_fact = result_factual.H_whitened
H_conf = result_confab.H_whitened

# Reference direction: query -> context (first token -> last token at mid layer)
mid_l = H_fact.shape[0] // 2
c_q = H_fact[mid_l, 0]; c_q = c_q / (np.linalg.norm(c_q) + 1e-12)
c_ctx = H_fact[mid_l, -1]; c_ctx = c_ctx / (np.linalg.norm(c_ctx) + 1e-12)

# Compute BCcos at each layer for both prompts (last token, consecutive layers)
bccos_fact = []
bccos_conf = []
for l in range(H_fact.shape[0] - 1):
    u_i = H_fact[l, -1]; u_i = u_i / (np.linalg.norm(u_i) + 1e-12)
    u_next = H_fact[l+1, -1]; u_next = u_next / (np.linalg.norm(u_next) + 1e-12)
    bccos_fact.append(binet_cauchy_cosine(u_i, u_next, c_q, c_ctx))

for l in range(H_conf.shape[0] - 1):
    u_i = H_conf[l, -1]; u_i = u_i / (np.linalg.norm(u_i) + 1e-12)
    u_next = H_conf[l+1, -1]; u_next = u_next / (np.linalg.norm(u_next) + 1e-12)
    bccos_conf.append(binet_cauchy_cosine(u_i, u_next, c_q, c_ctx))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(bccos_fact, 'o-', label='Factual', color='#2196F3', markersize=4)
ax.plot(bccos_conf, 's-', label='Confabulation', color='#f44336', markersize=4)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Layer transition')
ax.set_ylabel('BCcos')
ax.set_title('Binet–Cauchy Cosine: Reasoning Direction Alignment')
ax.legend()
ax.fill_between(range(len(bccos_fact)), 0, [-0.3]*len(bccos_fact),
                alpha=0.05, color='red', label='_causal inversion zone')
plt.tight_layout()
plt.show()

n_inversions_fact = sum(1 for x in bccos_fact if x < -0.1)
n_inversions_conf = sum(1 for x in bccos_conf if x < -0.1)
print(f"\nLayers with BCcos < -0.1 (potential causal inversions):")
print(f"  Factual:       {n_inversions_fact}/{len(bccos_fact)}")
print(f"  Confabulation: {n_inversions_conf}/{len(bccos_conf)}")
print("\n→ Strongly negative BCcos = causal inversion (reasoning flows backwards)."
      "\n  This oriented measure is invisible to scalar similarity.")

### Experiment: Factual vs. Confabulation Prompt

To see the "busy but useless" signature clearly, let's compare the full GA profile for a **factual** prompt (grounded in real knowledge) and a **confabulation** prompt (likely to produce fabricated content). If the confabulation prompt shows higher geometric activity but lower dependency, that confirms the model is churning through representations without arriving at a grounded answer.

In [ ]:
# --- Factual vs Confabulation: full GA profile comparison ---
from layer_time_ga.capacity import ga_capacity_profile

prompt_factual = "The Eiffel Tower is a wrought iron lattice tower in Paris France"
prompt_confab = "The underwater city of Atlantis was discovered in 2019 near the coast of"

r_fact = ltg_ga.analyse(prompt_factual, model=model)
r_conf = ltg_ga.analyse(prompt_confab, model=model)

rf_fact = r_fact.rotor_field
rf_conf = r_conf.rotor_field
cap_fact = ga_capacity_profile(r_fact.H_whitened)
cap_conf = ga_capacity_profile(r_conf.H_whitened)

# Summary table
print("=" * 60)
print(f"{'Metric':<25s} {'Factual':>12s} {'Confabulation':>15s}")
print("=" * 60)
print(f"{'Mean rotation angle':<25s} {rf_fact.angles.mean():12.2f} {rf_conf.angles.mean():15.2f}")
print(f"{'Max rotation angle':<25s} {rf_fact.angles.max():12.2f} {rf_conf.angles.max():15.2f}")
print(f"{'D_total':<25s} {r_fact.dep_total:12.2f} {r_conf.dep_total:15.2f}")
print(f"{'H(D) entropy':<25s} {r_fact.dep_entropy:12.2f} {r_conf.dep_entropy:15.2f}")
print(f"{'C_acc':<25s} {cap_fact.C_acc:12.1f} {cap_conf.C_acc:15.1f}")
print(f"{'Concentration':<25s} {cap_fact.cconc:12.3f} {cap_conf.cconc:15.3f}")
print("=" * 60)
print()

ratio = r_fact.dep_total / (r_conf.dep_total + 1e-12)
print(f"Key finding: Confabulation has {rf_conf.angles.mean()/rf_fact.angles.mean():.1f}x "
      f"higher mean rotation")
print(f"  but {ratio:.1f}x LOWER total dependency")
print(f"  => Classic 'busy but useless' signature")

# 4-panel comparison plot
D_fact = r_fact.dependency_profile
D_conf = r_conf.dependency_profile
n = min(len(D_fact), len(rf_fact.angles), len(D_conf), len(rf_conf.angles))
ew_fact = D_fact[:n] * rf_fact.angles[:n]
ew_conf = D_conf[:n] * rf_conf.angles[:n]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Top-left: rotation angles
axes[0,0].plot(rf_fact.angles, color='#2E6DAD', linewidth=2, label='Factual')
axes[0,0].plot(rf_conf.angles, color='#EE6677', linewidth=2, label='Confabulation')
axes[0,0].set_xlabel('Layer'); axes[0,0].set_ylabel('Rotation angle (rad)')
axes[0,0].set_title('Rotor Angle Profile'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# Top-right: dependency
axes[0,1].bar(np.arange(n)-0.15, D_fact[:n], 0.3, color='#2E6DAD', alpha=0.7, label='Factual')
axes[0,1].bar(np.arange(n)+0.15, D_conf[:n], 0.3, color='#EE6677', alpha=0.7, label='Confabulation')
axes[0,1].set_xlabel('Layer'); axes[0,1].set_ylabel(r'Dependency $D_l$')
axes[0,1].set_title('Dependency Profile'); axes[0,1].legend(); axes[0,1].grid(True, axis='y', alpha=0.3)

# Bottom-left: effective work
axes[1,0].bar(np.arange(n)-0.15, ew_fact, 0.3, color='#2E6DAD', alpha=0.7, label='Factual')
axes[1,0].bar(np.arange(n)+0.15, ew_conf, 0.3, color='#EE6677', alpha=0.7, label='Confabulation')
axes[1,0].set_xlabel('Layer'); axes[1,0].set_ylabel(r'Effective work $D_l \times \theta_l$')
axes[1,0].set_title('Effective Work'); axes[1,0].legend(); axes[1,0].grid(True, axis='y', alpha=0.3)

# Bottom-right: capacity
axes[1,1].plot(cap_fact.layer_contributions, color='#2E6DAD', linewidth=2, label='Factual')
axes[1,1].plot(cap_conf.layer_contributions, color='#EE6677', linewidth=2, label='Confabulation')
axes[1,1].set_xlabel('Layer pair'); axes[1,1].set_ylabel(r'$\|[B_l, B_{l+1}]\|_F$')
axes[1,1].set_title('Per-Layer Capacity'); axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("\nThe confabulation prompt shows higher rotation and capacity (busy)")
print("but much lower dependency (useless) — the model churns without grounding.")

## Application 3: Steering Targets

Representation engineering and activation steering are increasingly popular techniques for modifying model behavior at inference time. The basic idea is to add a **steering vector** $\mathbf{v}$ to the hidden state at some layer $\ell$:

$$\tilde{h}^{(\ell)} = h^{(\ell)} + \alpha \, \mathbf{v}$$

where $\alpha$ controls the strength of the intervention. The challenge is choosing *which layer* and *which direction* to intervene in.

**The standard approach** selects the layer with the highest causal effect (via gradient-based dependency or activation patching) and uses a steering vector derived from contrastive pairs (e.g., the mean difference between "truthful" and "untruthful" activations).

**The GA approach** refines this in an important way: instead of modifying all directions at a layer (which is a rank-$k$ intervention in $k$-dimensional space), we identify the **principal rotation plane** at the target layer and intervene only within that 2-dimensional subspace. This is a **rank-2 intervention** — far more targeted.

The concept of **effective work** helps us rank candidate intervention targets. The effective work at layer $\ell$ combines:

1. **Rotation magnitude** — how much the layer rotates the representation (rotor angle $\theta^{(\ell)}$)
2. **Functional relevance** — how much this rotation matters for the output (dependency $D_\ell$)
3. **Plane specificity** — how concentrated the rotation is in a single plane (ratio $\sigma_1 / \sigma_2$ of the top two singular values of the bivector)

$$W_{\text{eff}}^{(\ell)} = \theta^{(\ell)} \cdot D_\ell \cdot \frac{\sigma_1^{(\ell)}}{\sigma_1^{(\ell)} + \sigma_2^{(\ell)}}$$

Layers with high effective work are ideal steering targets: they rotate strongly in a specific plane, and that rotation matters for the output.

In [ ]:
# --- Identify top steering targets based on effective work ---

n_biv = len(result.bivector_field)
effective_work = np.zeros(n_biv)
plane_specificity = np.zeros(n_biv)
steering_info = []

for l in range(n_biv):
    biv = result.bivector_field[l]
    
    # Rotation angle
    theta = result.angles[l] if l < len(result.angles) else 0.0
    
    # Dependency (align indices: bivector field may start after layer 0)
    d_l = 0.0
    if result.dependency_profile is not None and l < len(result.dependency_profile):
        d_l = result.dependency_profile[l]
    
    # Plane specificity: extract top-2 principal planes
    planes = biv.principal_planes(n_planes=2)
    if len(planes) >= 2:
        s1 = planes[0]['weight']
        s2 = planes[1]['weight']
        specificity = s1 / (s1 + s2 + 1e-12)
    elif len(planes) == 1:
        specificity = 1.0  # perfectly simple rotation
    else:
        specificity = 0.0
    
    plane_specificity[l] = specificity
    effective_work[l] = theta * d_l * specificity
    
    steering_info.append({
        'layer': l,
        'angle': theta,
        'dependency': d_l,
        'specificity': specificity,
        'effective_work': effective_work[l],
        'planes': planes,
    })

# Rank by effective work and show top-2 targets
ranked = sorted(steering_info, key=lambda x: x['effective_work'], reverse=True)

print("=== Top-2 Steering Targets (by Effective Work) ===")
print()
for rank, info in enumerate(ranked[:2]):
    print(f"--- Target #{rank + 1}: Layer {info['layer']} ---")
    print(f"  Rotation angle:   {info['angle']:.4f} rad ({np.degrees(info['angle']):.1f} deg)")
    print(f"  Dependency:       {info['dependency']:.6f}")
    print(f"  Plane specificity: {info['specificity']:.3f}")
    print(f"  Effective work:   {info['effective_work']:.6f}")
    
    if info['planes']:
        p = info['planes'][0]
        v1, v2 = p['plane_vectors']
        print(f"  Principal plane weight: {p['weight']:.4f}")
        print(f"  Plane vector 1 (first 5 components): {v1[:5].round(4)}")
        print(f"  Plane vector 2 (first 5 components): {v2[:5].round(4)}")
        print(f"  Recommended intervention: project steering vector onto")
        print(f"    the plane spanned by v1 and v2 at this layer.")
    print()

# Visualize effective work across all layers
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: effective work profile
ax1.bar(range(n_biv), effective_work, color='#4477AA', edgecolor='white', linewidth=0.5)
for r in ranked[:2]:
    ax1.bar(r['layer'], r['effective_work'], color='#EE6677', edgecolor='white', linewidth=0.5)
ax1.set_xlabel('Layer transition')
ax1.set_ylabel('Effective work $W_{\\mathrm{eff}}$')
ax1.set_title('Effective Work Profile (top-2 targets highlighted)')
ax1.grid(True, axis='y', alpha=0.3)

# Right: plane specificity profile
ax2.plot(range(n_biv), plane_specificity, color='#228833', linewidth=2,
         marker='o', markersize=3)
ax2.axhline(0.5, color='grey', linestyle='--', alpha=0.5, label='Simple rotation threshold')
ax2.set_xlabel('Layer transition')
ax2.set_ylabel('Plane specificity $\\sigma_1 / (\\sigma_1 + \\sigma_2)$')
ax2.set_title('Rotation Plane Specificity by Layer')
ax2.set_ylim(0.4, 1.05)
ax2.legend()
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()
print("Layers with high effective work and high specificity are ideal rank-2 steering targets.")

## The GA Summary Plot

Let us put everything together in a single four-panel visualization. The `plot_ga_summary()` method on a `GAResult` object generates the canonical summary:

1. **Top-left: Rotor angles** — the magnitude of rotation at each layer transition
2. **Top-right: Holonomy curvature** — where the layer-time grid is geometrically curved (curvature by layer, averaged over tokens)
3. **Bottom-left: Metric deformation** — condition number $\kappa$ (selectivity) and effective rank (dimensionality) of the stretch component
4. **Bottom-right: Commutator heatmap** — $\|[B_i, B_j]\|_F$ showing which layer pairs fail to commute (and thus contribute to compositional capacity)

This single figure captures the essential GA story of any prompt: how much each layer rotates, where curvature concentrates, how the metric deforms, and how layers interact through non-commutativity.

In [ ]:
# --- Generate the 4-panel GA summary plot ---
save_path = os.path.join(os.getcwd(), 'ch13_ga_summary.png')
result.plot_ga_summary(save_path=save_path)
print(f"\nGA summary plot saved to: {save_path}")
print("This single figure captures the rotor, curvature, metric, and commutator")
print("structure of the model's internal geometry for the given prompt.")

## Exercises

**Exercise 13.1: Context sensitivity across layers.**
Run the context-ignoring detection experiment (Application 1) with a question where the context *contradicts* parametric knowledge. For example, provide a passage that says "The capital of France is Lyon" and ask "The capital of France is". At which layers does the bivector cosine similarity drop lowest? These are the layers where context override happens.

**Exercise 13.2: Hallucination detection on a fabricated claim.**
Replace the prompt in Application 2 with a prompt that asks about a non-existent entity (e.g., "The Zorbanian Principle of thermodynamics states that"). Compare the hallucination signature to a prompt about a real concept. Do you see more busy-but-useless layers for the fabricated claim?

**Exercise 13.3: Steering vector projection.**
Take the top steering target from Application 3. Construct a steering vector by taking the difference of hidden states between two contrasting prompts (e.g., "The answer is true" vs "The answer is false") at that layer. Project this steering vector onto the principal rotation plane using:

$$\mathbf{v}_{\text{proj}} = (\mathbf{v} \cdot \mathbf{u}_1) \mathbf{u}_1 + (\mathbf{v} \cdot \mathbf{u}_2) \mathbf{u}_2$$

where $\mathbf{u}_1, \mathbf{u}_2$ are the plane vectors. How much of the steering vector's norm is captured by this rank-2 projection? If the ratio $\|\mathbf{v}_{\text{proj}}\| / \|\mathbf{v}\|$ is high, the GA plane is well-aligned with the behavioral direction.

**Exercise 13.4: Multi-prompt consistency.**
Run the full analysis on 5 different prompts with the same thematic content (e.g., geography questions). Compare the effective work profiles. Do the same layers consistently appear as top steering targets, or does the target vary with the specific prompt? This tells you whether steering is a *model property* or a *prompt-dependent phenomenon*.

**Exercise 13.5: Commutator-based context detection.**
Instead of comparing bivectors at a single layer (Application 1), compare the *commutator matrices* $\|[B_i, B_j]\|_F$ between with-context and without-context conditions. Which layer *pairs* show the largest commutator change? This identifies interactions between layers that are context-sensitive.